In [3]:
!pip install mysql-connector-python pandas tqdm

In [4]:
import pandas as pd
import mysql.connector
from tqdm import tqdm

In [5]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Ananya@3283",
    database="ecommerce_sales"
)

cursor = conn.cursor()

print("✅ Connected Successfully!")

✅ Connected Successfully!


In [6]:
csv_path = r"C:\Users\ANANYA\OneDrive\Documents\Ecommerce_Customer_Sales_Analytics\Dataset\Cleaned\online_retail_cleaned.csv"

df = pd.read_csv(
    csv_path,
    low_memory=False,
    dtype={"Invoice": str}
)

print(df.shape)
df.head()

(505193, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [7]:
# Convert InvoiceDate to MySQL DATETIME format
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["InvoiceDate"] = df["InvoiceDate"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Replace missing values with None
df = df.where(pd.notnull(df), None)

print("Data prepared successfully!")

Data prepared successfully!


In [8]:
cursor.execute("TRUNCATE TABLE online_retail")
conn.commit()

print("Table cleared!")

Table cleared!


In [9]:
query = """
INSERT INTO online_retail
(
    Invoice,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    Price,
    CustomerID,
    Country,
    Revenue
)
VALUES
(
    %s,%s,%s,%s,%s,%s,%s,%s,%s
)
"""

data = list(df.itertuples(index=False, name=None))

batch_size = 5000

for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i + batch_size]
    cursor.executemany(query, batch)
    conn.commit()

print("✅ Import Completed Successfully!")

100%|██████████| 102/102 [00:40<00:00,  2.49it/s]

✅ Import Completed Successfully!


In [10]:
cursor.execute("SELECT COUNT(*) FROM online_retail")

rows = cursor.fetchone()

print("Total Rows Imported:", rows[0])

Total Rows Imported: 505193


In [11]:
cursor.execute("SELECT * FROM online_retail LIMIT 5")

for row in cursor.fetchall():
    print(row)

('489434', '85048', '15CM CHRISTMAS GLASS BALL 20 LIGHTS', 12, datetime.datetime(2009, 12, 1, 7, 45), Decimal('6.95'), 13085, 'United Kingdom', Decimal('83.40'))
('489434', '79323P', 'PINK CHERRY LIGHTS', 12, datetime.datetime(2009, 12, 1, 7, 45), Decimal('6.75'), 13085, 'United Kingdom', Decimal('81.00'))
('489434', '79323W', ' WHITE CHERRY LIGHTS', 12, datetime.datetime(2009, 12, 1, 7, 45), Decimal('6.75'), 13085, 'United Kingdom', Decimal('81.00'))
('489434', '22041', 'RECORD FRAME 7" SINGLE SIZE ', 48, datetime.datetime(2009, 12, 1, 7, 45), Decimal('2.10'), 13085, 'United Kingdom', Decimal('100.80'))
('489434', '21232', 'STRAWBERRY CERAMIC TRINKET BOX', 24, datetime.datetime(2009, 12, 1, 7, 45), Decimal('1.25'), 13085, 'United Kingdom', Decimal('30.00'))


In [12]:
cursor.close()
conn.close()

print("Connection Closed!")

Connection Closed!


In [16]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Ananya@3283",
    database="ecommerce_sales"
)

cursor = conn.cursor()

print("Connected:", conn.is_connected())

Connected: True


In [17]:
cursor.execute("""
INSERT INTO online_retail
(Invoice, StockCode, Description, Quantity, InvoiceDate, Price, CustomerID, Country, Revenue)
VALUES
('TEST001','ABC123','Test Product',1,'2024-01-01 10:00:00',10.50,1,'India',10.50)
""")

conn.commit()

cursor.execute("SELECT COUNT(*) FROM online_retail")
print(cursor.fetchone())

(1,)


In [18]:
print(df.dtypes)
print(df.isnull().sum())
print(df.head())

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
Revenue        float64
dtype: object
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    104246
Country             0
Revenue             0
dtype: int64
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  Revenue  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom     83.4  
1  2009

In [20]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Ananya@3283",
    database="ecommerce_sales"
)

cursor = conn.cursor()

# Read CSV
df = pd.read_csv(
    r"C:\Users\ANANYA\OneDrive\Documents\Ecommerce_Customer_Sales_Analytics\Dataset\Cleaned\online_retail_sql.csv",
    low_memory=False
)

# Clean data
df["Customer ID"] = df["Customer ID"].fillna(0).astype(int)
df["Description"] = df["Description"].fillna("")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"]).dt.strftime("%Y-%m-%d %H:%M:%S")

# Clear table
cursor.execute("TRUNCATE TABLE online_retail")

query = """
INSERT INTO online_retail
(Invoice, StockCode, Description, Quantity, InvoiceDate,
 Price, CustomerID, Country, Revenue)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

data = list(zip(
    df["Invoice"].astype(str),
    df["StockCode"].astype(str),
    df["Description"],
    df["Quantity"].astype(int),
    df["InvoiceDate"],
    df["Price"].astype(float),
    df["Customer ID"].astype(int),
    df["Country"],
    df["Revenue"].astype(float)
))

cursor.executemany(query, data)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM online_retail")
print("Rows Imported:", cursor.fetchone()[0])

cursor.close()
conn.close()

Rows Imported: 505193
